In [18]:
import zipfile
import pandas as pd
from pathlib import Path
import regex
import unicodedata

In [ ]:
diretorio = '.'

In [20]:
groundtruth_union = f'{diretorio}/opendata_union_ground_truth.csv'
groundtruth_join = f'{diretorio}/opendata_join_ground_truth.csv'

In [21]:
df_union = pd.read_csv(groundtruth_union)
df_union['country'] = df_union['query_table'].str.split('_', n=1).str[0]
df_union

,query_table,candidate_table,country
0,CAN_CSV0000000000000474.csv,CAN_CSV0000000000004972.csv,CAN
1,CAN_CSV0000000000000474.csv,CAN_CSV0000000000004975.csv,CAN
2,CAN_CSV0000000000000474.csv,CAN_CSV0000000000000471.csv,CAN
3,CAN_CSV0000000000000474.csv,CAN_CSV0000000000004970.csv,CAN
4,CAN_CSV0000000000000474.csv,CAN_CSV0000000000000474.csv,CAN
...,...,...,...
61975,USA_CSV0000000000066309__53.csv,USA_CSV0000000000011201__89.csv,USA
61976,USA_CSV0000000000076228__0.csv,USA_CSV0000000000014965__0.csv,USA
61977,USA_CSV0000000000078506__9.csv,USA_CSV0000000000023906__8.csv,USA
61978,USA_CSV0000000000080056__48.csv,USA_CSV0000000000036044__49.csv,USA


In [22]:
df_join = pd.read_csv(groundtruth_join, usecols=['query_table','candidate_table'])
df_join['country'] = df_join['query_table'].str.split('_', n=1).str[0]
df_join

,query_table,candidate_table,country
0,CAN_CSV0000000000001724__13.csv,CAN_CSV0000000000001787__9.csv,CAN
1,UK_CSV0000000000003896__7.csv,UK_CSV0000000000002010__11.csv,UK
2,UK_CSV0000000000003896__7.csv,UK_CSV0000000000002010__11.csv,UK
3,USA_CSV0000000000037101__16.csv,USA_CSV0000000000033847__23.csv,USA
4,USA_CSV0000000000037101__16.csv,USA_CSV0000000000033847__23.csv,USA
...,...,...,...
42839,USA_CSV0000000000095808.csv,USA_CSV0000000000010779.csv,USA
42840,USA_CSV0000000000096676.csv,USA_CSV0000000000005360.csv,USA
42841,USA_CSV0000000000098075.csv,USA_CSV0000000000012031.csv,USA
42842,USA_CSV0000000000098620.csv,USA_CSV0000000000048479.csv,USA


In [23]:
df_comparacao = pd.concat([df_join, df_union], ignore_index=True).drop_duplicates()
df_comparacao

,query_table,candidate_table,country
0,CAN_CSV0000000000001724__13.csv,CAN_CSV0000000000001787__9.csv,CAN
1,UK_CSV0000000000003896__7.csv,UK_CSV0000000000002010__11.csv,UK
3,USA_CSV0000000000037101__16.csv,USA_CSV0000000000033847__23.csv,USA
10,USA_CSV0000000000033847__5.csv,USA_CSV0000000000034882__27.csv,USA
15,UK_CSV0000000000003896__0.csv,UK_CSV0000000000007396.csv,UK
...,...,...,...
104819,USA_CSV0000000000066309__53.csv,USA_CSV0000000000011201__89.csv,USA
104820,USA_CSV0000000000076228__0.csv,USA_CSV0000000000014965__0.csv,USA
104821,USA_CSV0000000000078506__9.csv,USA_CSV0000000000023906__8.csv,USA
104822,USA_CSV0000000000080056__48.csv,USA_CSV0000000000036044__49.csv,USA


In [ ]:
zips = [{'pais':'USA',
         'arq':'datasets_USA.zip'},
        {'pais':'CAN',
         'arq':'datasets_CAN.zip'},
        {'pais':'UK',
         'arq':'datasets_UK.zip'}]

In [ ]:
def padronizar_coluna(colunas_cabecalho):
    lista_padronizada = []
    
    for texto in colunas_cabecalho:
        
        texto = texto.lower().strip() # Trasnformar para minúsculo e remover espaços no ínicio e fim
        texto = ''.join(regex.findall(r'[a-zA-ZÀ-ÿ0-9]', texto)) # Remover caracteres especiais e espaços internos
        texto_normalizado = unicodedata.normalize('NFKD', texto) # Normaliza o texto para separar acentos das letras
        texto = ''.join(c for c in texto_normalizado if not unicodedata.combining(c)) # Remove os caracteres que não são ASCII (ou seja, os acentos)
        
        lista_padronizada.append(texto)
    
    return set(lista_padronizada)

In [ ]:
def comparar_arquivos(source, compare):
    source_p = padronizar_coluna(source)
    compare_p = padronizar_coluna(compare)
    
    colunas_em_ambos = source_p & compare_p
    colunas_diferentes = source_p ^ compare_p
    
    return len(colunas_diferentes) > 0 and len(colunas_em_ambos) > 0

In [ ]:
matchs = []
for z in zips:
    arquivo = z['arq']
    caminho = f'{diretorio}/{arquivo}'
    try:
        with zipfile.ZipFile(caminho) as archive:
            for i,r in df_comparacao[df_comparacao.country == z['pais']].iterrows():
                df_source = None
                df_compare = None
                file_source = None
                file_compare = None
                for filename in archive.namelist()[1:]:
                    # print(filename)
                    caminho_path = Path(filename)
                    if caminho_path.suffix == '.csv':
                        csv = caminho_path.stem + caminho_path.suffix
                        if csv == r['query_table']:
                            with archive.open(filename)  as f:
                                df_source = pd.read_csv(f, nrows=0)
                                df_source = list(df_source.columns)
                                file_source = filename
                        elif csv == r['candidate_table']:
                            with archive.open(filename)  as f:
                                df_compare = pd.read_csv(f, nrows=0)
                                df_compare = list(df_compare.columns)
                                file_compare = filename
                        if df_compare != None and df_source != None:
                            break
                        # print(f"Nome base: {caminho_path.stem} {caminho+filename}/ Extensão: {caminho_path.suffix}")
                # comparar cabecalhos!!!
                if df_compare != None and df_source != None:
                    if comparar_arquivos(df_source, df_compare):
                        matchs.append(
                            {
                                'source':file_source,
                                'compare':file_compare
                            }
                        )
                        
    except zipfile.BadZipFile as error:
        print('error',error)
    except FileNotFoundError:
        print(f"Erro: Arquivo '{arquivo}' não encontrado no diretório '{diretorio}'.")
    except Exception as e:
        print(f"Ocorreu um erro inesperado: {e}")
        
if len(matchs) > 0:
    df = pd.DataFrame(matchs)
    df.to_csv('./matching_lakebench.csv', index=False)

datasets_USA/USA_CSV0000000000000373__10.csv
Nome base: USA_CSV0000000000000373__10 C:/Users/user/Desktop/bolsa estudo/meterial estudo/artigos agosto/LakeBench/datasets_USA.zipdatasets_USA/USA_CSV0000000000000373__10.csv/ Extensão: .csv
